# Create weather-dependent smartds files (at peak demand timestep)

This notebook does the following:
1. Define path to opendss ditribution data region folder
2. Load measured and predicted electricity buildings demand data
3. Convert building kw timeseries to smart-ds format kw and kvar loadshapes and max values
4. Extract temp at each timestep (hourly)
5. Create smart-ds LineCodes.dss and Transformers.dss using temp at each timestep
6. Create smart-ds csv profiles, Loads.dss, LoadShapes.dss, PVSystems.dss files using buildings demand predictions and weather data
7. Create master.dss file with paths to new load.dss and loadshapes.dss + LineCodes.dss and Transformers.dss

## Initialize files

### Import packages

In [3]:
import time
import datetime
import os
import numpy as np 
import glob
import re
import shutil
import pandas as pd
import joblib
import yaml
import matplotlib.pyplot as plt
import math
import gc
import sys
from collections import defaultdict

from opendssdirect import dss
from src import input_ops
from src import opendss_ops
from src import file_ops
from src import df_ops

### Define functions

In [3]:
def load_region_building_inputs(
    config,
    city,
    region,
    TGW_scenario,
    TGW_weather_year,
    demand_mode,
):
    """
    Load measured buildings' demand data and predicted buildings' total demand
    timeseries for one city-region.
    """

    smart_ds_year = config["smart_ds_years"][0]
    smart_ds_load_path = config["smart_ds_load_path"] + f"/{smart_ds_year}"

    # Load measured buildings' demand timeseries dataframe, organized by feeders
    # Columns include:
    # total_site_electricity_kw, pf, cooling_sum_kw, heating_kw, non_cool_n_heat_kw
    input_data_region_dir = f"{smart_ds_load_path}/{city}/{region}/buildings"

    measured_buildings_total_kw_pf_dict = joblib.load(
        os.path.join(
            input_data_region_dir,
            "measured_buildings_total_kw_pf_dict.joblib", 	
        )
    )

    # Load predicted buildings' total demand pandas timeseries, organized by feeders
    predictions_path = (
        "main_folder/load_prediction/"
        f"results/data/prediction/output/{smart_ds_year}/"
        f"months_{config['start_month']}_{config['end_month']}/"
        f"cooling_n_heating/{config['X_columns_set']}/"
        f"{config['aggregation_level']}/"
    )

    predictions_dir = os.path.join(
        predictions_path,
        f"{TGW_scenario}/predictions/{city}/{region}/",
    )

    building_predicted_total_dict = joblib.load(
        os.path.join(
            predictions_dir,
            f"{demand_mode}_TGW_{TGW_weather_year}_buildings_dict.joblib",
        )
    )

    return measured_buildings_total_kw_pf_dict, building_predicted_total_dict




def add_predicted_kvar_from_measured_pf(
    building_predicted_total_dict,
    measured_buildings_total_kw_pf_dict,
):
    """
    Create predicted reactive load profiles (kvar) using predicted kw
    and measured power factor time series.

    Each resulting dataframe has columns:
    kw, kvar
    """

    # Create predicted kvar from predicted kw and measured pf
    for outer_key, inner_dict in building_predicted_total_dict.items():
        for building_name, kw_series in inner_dict.items():
            # Convert active power series to DataFrame
            df = kw_series.to_frame(name="kw")

            # Get the matching power factor series
            pf_series = measured_buildings_total_kw_pf_dict[outer_key][building_name]["pf"]

            # Align indices if needed
            pf_series = pf_series.loc[df.index]

            # Calculate reactive power
            angle_rad = np.arccos(pf_series.clip(lower=0.01, upper=1.0))
            df["kvar"] = df["kw"] * np.tan(angle_rad)

            # Store the result
            building_predicted_total_dict[outer_key][building_name] = df

    return building_predicted_total_dict


def compute_measured_building_max_dict(measured_buildings_total_kw_pf_dict):
    """
    Use measured kw and measured power factor time series to create measured
    kw and kvar max values per building.
    """

    # Initialize dictionary to store max values
    building_measured_max_dict = {}

    for outer_key, inner_dict in measured_buildings_total_kw_pf_dict.items():
        building_measured_max_dict[outer_key] = {}

        for building_name, df in inner_dict.items():
            # Compute kvar timeseries
            # Get the matching power factor series
            pf_series = measured_buildings_total_kw_pf_dict[outer_key][building_name]["pf"]

            # Align indices if needed
            pf_series = pf_series.loc[df.index]

            # Calculate reactive power
            angle_rad = np.arccos(pf_series.clip(lower=0.01, upper=1.0))

            total_site_electricity_kvar = (
                df["total_site_electricity_kw"] * np.tan(angle_rad)
            )

            # Compute max values
            kw_max = df["total_site_electricity_kw"].max()
            kvar_max = total_site_electricity_kvar.max()

            # Store max values
            building_measured_max_dict[outer_key][building_name] = {
                "kw_max": kw_max,
                "kvar_max": kvar_max,
            }

    return building_measured_max_dict


def get_reference_calendar_arrays(building_predicted_total_dict):
    """
    Get lists of months, days, and hours from a sample dataframe in
    building_predicted_total_dict. These are later used to mask selected mdh.

    Assumption:
    all building dataframes have the same full-year hourly timestamp index.
    """

    first_outer_key, first_inner_dict = next(iter(building_predicted_total_dict.items()))
    first_building_name, first_df = next(iter(first_inner_dict.items()))

    if not isinstance(first_df.index, pd.DatetimeIndex):
        first_df = first_df.copy()
        first_df.index = pd.to_datetime(first_df.index, errors="raise")

    months = first_df.index.month
    days = first_df.index.day
    hours = first_df.index.hour

    return months, days, hours


def build_feeder_dict(building_predicted_total_dict):
    """
    Organize all building_type entries under each feeder for Loads.dss processing.
    """

    feeder_dict = defaultdict(dict)

    for key, building_dict in building_predicted_total_dict.items():
        smart_ds_year, city, region, feeder, building_type = key
        feeder_key = (smart_ds_year, city, region, feeder)
        feeder_dict[feeder_key][building_type] = building_dict

    return feeder_dict


def get_temperature_for_mdh(TGW_weather_df, m, d, h):
    """
    Extract temperature at selected month-day-hour from TGW weather dataframe.
    """

    # Filter TGW_weather_df rows by matching month, day, and hour
    matched = TGW_weather_df[
        (TGW_weather_df["date_time"].dt.month == m)
        & (TGW_weather_df["date_time"].dt.day == d)
        & (TGW_weather_df["date_time"].dt.hour == h)
    ]

    # Extract temperature at m, d, h
    Ta = float(matched[["Dry Bulb Temperature [°C]"]].iloc[0, 0])

    return Ta

def get_irradiance_for_mdh(TGW_weather_df, m, d, h):
    """
    Extract global horizontal radiation at selected month-day-hour
    and convert from W/m2 to OpenDSS irradiance units.

    OpenDSS PVSystem irradiance is in kW/m2, so 1000 W/m2 -> irradiance=1
    """

    radiation_col = "Global Horizontal Radiation [W/m2]"

    if radiation_col not in TGW_weather_df.columns:
        raise KeyError(
            f"Column {radiation_col!r} was not found in TGW_weather_df. "
            f"Available columns: {list(TGW_weather_df.columns)}"
        )

    matched = TGW_weather_df[
        (TGW_weather_df["date_time"].dt.month == m)
        & (TGW_weather_df["date_time"].dt.day == d)
        & (TGW_weather_df["date_time"].dt.hour == h)
    ]

    if matched.empty:
        raise ValueError(f"No TGW weather row found for month={m}, day={d}, hour={h}")

    ghi_w_m2 = float(matched[radiation_col].iloc[0])

    # Convert W/m2 to kW/m2 for OpenDSS irradiance
    irradiance = ghi_w_m2 / 1000.0
    irradiance = max(irradiance, 0.0)

    return irradiance


def create_thermal_files_for_mdh(
    folders_with_linecodes,
    folders_with_transformers,
    Ta,
    Ta_near_worst,
    T0,
    alpha_r,
    f_amp_temp,
    line_rating_mode,
    transformers_rating_mode,
    TGW_scenario,
    TGW_weather_year,
    m,
    d,
    h,
):
    """
    Modify lines' thermal capacity and resistance in LineCodes files,
    and modify transformers' thermal capacity in Transformers files.
    """

    # Calculate resistance and ampacity temperature-based factors
    Rmatrix_factor = 1 + (alpha_r * (Ta - T0))
    amp_factor = 1 - ((Ta - Ta_near_worst) * f_amp_temp)

    # Modify lines' thermal capacity and resistance in LineCodes files
    for folder_path in folders_with_linecodes:
        opendss_ops.modify_LineCodes(
            folder_path,
            Rmatrix_factor,
            amp_factor,
            line_rating_mode,
            TGW_scenario,
            TGW_weather_year,
            m,
            d,
            h,
        )

    # Modify transformers' thermal capacity (KVA rating values)
    # Uses IEEE C57.91 Table 3 approximation
    for folder_path in folders_with_transformers:
        opendss_ops.modify_tranformers(
            folder_path,
            Ta,
            transformers_rating_mode,
            TGW_scenario,
            TGW_weather_year,
            m,
            d,
            h,
        )

    return {
        "Rmatrix_factor": Rmatrix_factor,
        "amp_factor": amp_factor,
    }


def find_feeder_folder(feeder_folders, feeder):
    """
    Use regex to find the exact feeder folder.
    """

    feeder_pattern = re.compile(rf"(^|/){re.escape(feeder)}(/|$)")
    matching_feeders = [f for f in feeder_folders if feeder_pattern.search(f)]

    if not matching_feeders:
        print(f"[WARNING] Feeder folder not found for feeder: {feeder}")
        return None

    return matching_feeders[0]


def parse_load_line(line, feeder):
    """
    Extract data from one Loads.dss line:
    - ResStock building/loadshape name
    - phase load name
    - measured phase-load kw/kvar values
    - building name and building type
    """

    # Extract resstock building name from loadShape name
    # Example: res_kw_452_pu
    # loadshape_pattern.group(1) = res
    # loadshape_pattern.group(2) = kw
    # loadshape_pattern.group(3) = 452
    loadshape_pattern = re.search(
        r"yearly=(\w+)_(kw|kvar)_(\d+)_pu",
        line,
    )

    # Extract load phase name, e.g., load_p1rlv5636_2
    phase_load_name_pattern = re.search(r"Load\.(\S+)\s", line)

    # Make sure patterns were found
    if not loadshape_pattern or not phase_load_name_pattern:
        raise ValueError("Failed to extract required fields from Loads.dss line")

    building_loadshape_name = (
        f"{loadshape_pattern.group(1)}_"
        f"{loadshape_pattern.group(2)}_"
        f"{loadshape_pattern.group(3)}_pu_"
        f"{feeder}"
    )

    phase_load_name = phase_load_name_pattern.group(1)

    # Extract measured kw/kvar load phase values
    pattern = (
        r"kW=([+-]?[0-9]*\.?[0-9]+(?:[eE][+-]?[0-9]+)?)"
        r"\s+kvar=([+-]?[0-9]*\.?[0-9]+(?:[eE][+-]?[0-9]+)?)"
    )

    match = re.search(pattern, line)

    if not match:
        raise ValueError("Failed to extract kW/kvar from Loads.dss line")

    measured_load_phase_kw_max = float(match.group(1))
    measured_load_phase_kvar_max = float(match.group(2))

    building_name = f"{loadshape_pattern.group(1)}_{loadshape_pattern.group(3)}"
    building_type = building_name.split("_")[0]

    return {
        "building_loadshape_name": building_loadshape_name,
        "phase_load_name": phase_load_name,
        "measured_load_phase_kw_max": measured_load_phase_kw_max,
        "measured_load_phase_kvar_max": measured_load_phase_kvar_max,
        "building_name": building_name,
        "building_type": building_type,
    }


def compute_predicted_phase_load_values(
    building_predicted_total_dict,
    building_measured_max_dict,
    outer_key,
    building_name,
    measured_load_phase_kw_max,
    measured_load_phase_kvar_max,
    mdh_mask,
):
    """
    Scale predicted ResStock building kw/kvar values to OpenDSS phase-load values.

    Logic:
    predicted phase load at mdh =
        predicted ResStock building value at mdh
        * measured OpenDSS phase-load max
        / measured ResStock building max
    """

    # Load measured ResStock kw and kvar max
    measured_ressstock_kw_max = (
        building_measured_max_dict[outer_key][building_name]["kw_max"]
    )

    measured_ressstock_kvar_max = (
        building_measured_max_dict[outer_key][building_name]["kvar_max"]
    )

    if measured_ressstock_kw_max == 0 or measured_ressstock_kvar_max == 0:
        raise ValueError(
            f"[ERROR] Zero measured max for {building_name} in {outer_key}"
        )


    # Compute kw/kvar values at selected mdh
    building_df = building_predicted_total_dict[outer_key][building_name]

    kw_vals = building_df.loc[mdh_mask, "kw"]
    kvar_vals = building_df.loc[mdh_mask, "kvar"]

    predicted_ressstock_kw_at_mdh = kw_vals.iloc[0]
    predicted_ressstock_kvar_at_mdh = kvar_vals.iloc[0]


    predicted_load_phase_kw_at_mdh = (
        predicted_ressstock_kw_at_mdh
        * (measured_load_phase_kw_max / measured_ressstock_kw_max)
    )

    predicted_load_phase_kvar_at_mdh = (
        predicted_ressstock_kvar_at_mdh
        * (measured_load_phase_kvar_max / measured_ressstock_kvar_max)
    )

    return {
        "predicted_kw_at_mdh": predicted_load_phase_kw_at_mdh,
        "predicted_kvar_at_mdh": predicted_load_phase_kvar_at_mdh,
    }


def update_load_line_for_predicted_values(
    line,
    predicted_load_phase_kw,
    predicted_load_phase_kvar,
    feeder,
):
    """
    Modify Loads.dss line with predicted kw/kvar values and new ResStock
    load shape names by adding feeder name to existing yearly= field.
    """

    # Update kW and kvar values in the line
    line = re.sub(
        r"kW=([0-9\.]+)",
        lambda m: f"kW={predicted_load_phase_kw}",
        line,
    )

    line = re.sub(
        r"kvar=([0-9\.]+)",
        lambda m: f"kvar={predicted_load_phase_kvar}",
        line,
    )

    # Add feeder name to existing yearly= field
    line = re.sub(r"(yearly=[^\s\n]+)", r"\1_" + feeder, line)

    return line

def create_loads_files_for_mdh(
    feeder_dict,
    feeder_folders,
    building_predicted_total_dict,
    building_measured_max_dict,
    TGW_scenario,
    TGW_weather_year,
    m,
    d,
    h,
    mdh_mask,
):
    """
    Create modified Loads.dss files for selected mdh.

    For each feeder:
    - create scenario-based path
    - duplicate Loads.dss
    - replace each phase load's kw/kvar values with predicted mdh values
    """

    # Iterate over each feeder once
    for feeder_key, building_types_dict in feeder_dict.items():
        smart_ds_year, city, region, feeder = feeder_key

        # Use feeder_key and feeder_folders to get path to Loads.dss
        feeder_folder = find_feeder_folder(feeder_folders, feeder)

        if feeder_folder is None:
            continue

        # Path to original Loads.dss
        path_to_loads_dss = os.path.join(feeder_folder, "Loads.dss")

        # Create scenario-based path:
        # feeder_folder/predicted_loads/TGW/climate_scenario/weather_year/
        output_dir = os.path.join(
            feeder_folder,
            "predicted_loads",
            "TGW",
            TGW_scenario,
            TGW_weather_year,
        )

        os.makedirs(output_dir, exist_ok=True)

        original_loads_path = path_to_loads_dss
        new_loads_path = os.path.join(output_dir, f"Loads_{m}_{d}_{h}.dss")

        # Create copy of Loads.dss in scenario-based path
        shutil.copyfile(original_loads_path, new_loads_path)

        with open(original_loads_path, "r") as infile, open(new_loads_path, "w") as outfile:
            # Loop over all rows in Loads.dss (all phase loads)
            for line in infile:
                if line.startswith("New Load."):
                    parsed = parse_load_line(line, feeder)

                    phase_load_name = parsed["phase_load_name"]
                    building_name = parsed["building_name"]
                    building_type = parsed["building_type"]

                    outer_key = (
                        feeder_key[0],
                        feeder_key[1],
                        feeder_key[2],
                        feeder_key[3],
                        building_type,
                    )


                    predicted_values = compute_predicted_phase_load_values(
                        building_predicted_total_dict=building_predicted_total_dict,
                        building_measured_max_dict=building_measured_max_dict,
                        outer_key=outer_key,
                        building_name=building_name,
                        measured_load_phase_kw_max=parsed[
                            "measured_load_phase_kw_max"
                        ],
                        measured_load_phase_kvar_max=parsed[
                            "measured_load_phase_kvar_max"
                        ],
                        mdh_mask=mdh_mask,
                    )

                    predicted_load_phase_kw = predicted_values["predicted_kw_at_mdh"]
                    predicted_load_phase_kvar = predicted_values["predicted_kvar_at_mdh"]

                    line = update_load_line_for_predicted_values(
                        line=line,
                        predicted_load_phase_kw=predicted_load_phase_kw,
                        predicted_load_phase_kvar=predicted_load_phase_kvar,
                        feeder=feeder,
                    )

                # Modify Loads.dss in scenario-based path with the new line
                outfile.write(line)

    return

def create_pvsystems_files_for_mdh(
    folders_with_pvsystems,
    irradiance,
    TGW_scenario,
    TGW_weather_year,
    m,
    d,
    h,
):
    """
    Create modified PVSystems.dss files for selected mdh.

    For each feeder folder containing PVSystems.dss:
    - create scenario-based path:
        feeder_folder/predicted_pvsystems/TGW/{TGW_scenario}/{TGW_weather_year}/
    - create PVSystems_{m}_{d}_{h}.dss
    - replace all irradiance=<number> instances with irradiance={irradiance}
    """

    new_pvsystems_paths = {}

    irradiance_pattern = re.compile(
        r"(?i)(\birradiance\s*=\s*)"
        r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)"
    )

    for feeder_folder in folders_with_pvsystems:
        original_pvsystems_path = os.path.join(feeder_folder, "PVSystems.dss")

        if not os.path.isfile(original_pvsystems_path):
            print(f"[WARNING] PVSystems.dss not found in {feeder_folder}")
            continue

        output_dir = os.path.join(
            feeder_folder,
            "predicted_pvsystems",
            "TGW",
            TGW_scenario,
            TGW_weather_year,
        )

        os.makedirs(output_dir, exist_ok=True)

        new_pvsystems_path = os.path.join(
            output_dir,
            f"PVSystems_{m}_{d}_{h}.dss",
        )

        n_replacements = 0

        with open(original_pvsystems_path, "r") as infile, open(
            new_pvsystems_path, "w"
        ) as outfile:
            for line in infile:
                line_new, count = irradiance_pattern.subn(
                    rf"\g<1>{irradiance}",
                    line,
                )

                n_replacements += count
                outfile.write(line_new)

        if n_replacements == 0:
            print(
                f"[WARNING] No irradiance=<number> entries were replaced in "
                f"{original_pvsystems_path}"
            )

        new_pvsystems_paths[feeder_folder] = new_pvsystems_path

    return new_pvsystems_paths

def create_master_file_for_mdh(
    region_path,
    solution_mode,
    demand_mode,
    line_rating_mode,
    transformers_rating_mode,
    TGW_scenario,
    TGW_weather_year,
    m,
    d,
    h,
):
    """
    Create a duplicate master file with paths to new Loads.dss,
    LineCodes.dss, and Transformers.dss files.
    """

    original_master_file = os.path.join(region_path, "Master.dss")

    new_master_dir = os.path.join(
        region_path,
        "predicted_master_files",
        TGW_scenario,
        TGW_weather_year,
    )

    os.makedirs(new_master_dir, exist_ok=True)

    new_master_file = os.path.join(
        new_master_dir,
        f"Master_{TGW_scenario}_{TGW_weather_year}_{m}_{d}_{h}.dss",
    )

    # Duplicate master file
    shutil.copyfile(original_master_file, new_master_file)

    opendss_ops.modify_master_file(
        new_master_file,
        solution_mode,
        demand_mode,
        line_rating_mode,
        transformers_rating_mode,
        TGW_scenario,
        TGW_weather_year,
        m,
        d,
        h,
    )

    return new_master_file

### Load config file with scenarios and parameters 

In [1]:
config_file_name = "opendss_config1"; config_path = f"config/{config_file_name}.yaml"; config = input_ops.load_config(config_path)

TGW_years_scenarios = config["TGW_years_scenarios"]
CITY_REGIONS_TO_RUN = config["CITY_REGIONS_TO_RUN"]

demand_mode = config["demand_mode"]  
line_rating_mode = config["line_rating_mode"] 
transformers_rating_mode = config["transformers_rating_mode"]  

aggregation_level = config["aggregation_level"]

smart_ds_year = config["smart_ds_years"][0]
start_month = config["start_month"]
end_month = config["end_month"]

smart_ds_load_path = config["smart_ds_load_path"] + f"/{smart_ds_year}" 
input_data_prediction_path = config["input_data_prediction_path"]

solution_mode = config["solution_mode"]  

# Define variables to create list of mdh to run
start_month_mdh = config["start_month_mdh"]
end_month_mdh = config["end_month_mdh"]
top_percent_mdh = config["top_percent_mdh"]

# Define start and end load hours to run
start_row_percent = config["start_row_percent"]

top_n_hours = int(np.ceil(8760 * top_percent_mdh / 100))
start_row_idx = int(np.ceil(8760 * start_row_percent / 100))
end_row_idx = top_n_hours

# --- Define parameters for resistance model ---
T0 = 20  # default temperature assumed for resistance values in SMART-DS
alpha_r = 0.00403  # temperature coefficient for Aluminum (from Anders2005)
f_amp_temp = 0.012  # ampacity change per degree Celsius (%/C)

# Assumption for ampacity near-worst weather condition baseline. p99 is typically used by utilities and is the CIGRE guideline.
# Options: avg_daily_max_august | p98 | p99 | p995 | p999
near_worst_stat = "p99"

regional_peak_start_month = 6
regional_peak_end_month = 9

# Load dictionary, sort by total city aggregated buildings demand, extract mdh of top % load hours
regional_demand_weather_ampacity_all_cities_sorted = input_ops.load_and_sort_regional_demand(config)

# Solar / battery SMART-DS scenario parameters
solar_share = config.get("solar_share", "none")
battery_share = config.get("battery_share", "none")

solar_battery_scenario_folder = input_ops.build_solar_battery_scenario_folder(
    solar_share=solar_share,
    battery_share=battery_share,
)

print(
    f"TGW_years_scenarios: {TGW_years_scenarios} "
    f"\nsmart_ds_year:{smart_ds_year} "
    f"\nsolution_mode:{solution_mode} "
    f"\ncity region: {CITY_REGIONS_TO_RUN} "
    f"\n Demand mode: {demand_mode} "
    f"\nline_rating_mode: {line_rating_mode} "
    f"\ntransformers_rating_mode: {transformers_rating_mode}"
)

print(
    f"solar_share: {solar_share}\n"
    f"battery_share: {battery_share}\n"
    f"solar_battery_scenario_folder: {solar_battery_scenario_folder}"
)

print(
    f"\n\ntop_percent_mdh: {top_percent_mdh}%, "
    f"top_n_hours: {top_n_hours}, "
    f"start_row_idx:{start_row_idx} ({start_row_percent}%), "
    f"end_row_idx:{end_row_idx} ({top_percent_mdh}%) \n"
)

# Conversion script

In [5]:
start_time = time.time()


print(f"--- Solution mode: {solution_mode} Demand mode:{demand_mode} Line rating mode:{line_rating_mode} --- \n")

# Exit program if solution_mode is different than snapshot mode
if solution_mode != "snapshot":
    sys.stderr.write(f"ERROR: solution_mode must be 'snapshot', got {solution_mode!r}.\n")
    sys.exit(1)

for TGW_weather_year, TGW_scenarios in TGW_years_scenarios.items():
    for TGW_scenario in TGW_scenarios:
        print(f"--- TGW_scenario:{TGW_scenario} TGW_weather_year:{TGW_weather_year} --- \n")

        for city, regions in CITY_REGIONS_TO_RUN.items():
            city_inputs = input_ops.load_city_weather_inputs(
                config=config,
                city=city,
                TGW_scenario=TGW_scenario,
                TGW_weather_year=TGW_weather_year,
                regional_demand_weather_ampacity_all_cities_sorted=regional_demand_weather_ampacity_all_cities_sorted,
                smart_ds_year=smart_ds_year,
                near_worst_stat=near_worst_stat,
                top_n_hours=top_n_hours,
            )

            print(
                f"Near-worst ({near_worst_stat}) historical local "
                f"temperature found for TGW_scenario historical "
                f"TGW_weather_year {smart_ds_year} {city} is "
                f"{city_inputs['Ta_near_worst']} "
                f"(used as base temp for line derating)\n"
            )

            for region in regions:
                print(f"--- city: {city}, region: {region} ---\n")

                region_start_time = time.time()

                # Define path to OpenDSS region folder (assuming snapshot mode)
                region_path = (
                    f"main_folder/SMART-DS/v1.0/"
                    f"{smart_ds_year}/{city}/{region}/scenarios/"
                    f"{solar_battery_scenario_folder}/opendss_no_loadshapes"
                )
                
                # --- Load demand data: regional data at the building level ---
                measured_buildings_total_kw_pf_dict, building_predicted_total_dict = (
                    load_region_building_inputs(
                        config=config,
                        city=city,
                        region=region,
                        TGW_scenario=TGW_scenario,
                        TGW_weather_year=TGW_weather_year,
                        demand_mode=demand_mode,
                    )
                )

                # --- Pre-process data ---
                # Convert ResStock building kw timeseries to SMART-DS kw and kvar loadshapes and max values
                building_predicted_total_dict = add_predicted_kvar_from_measured_pf(
                    building_predicted_total_dict,
                    measured_buildings_total_kw_pf_dict,
                )

                building_measured_max_dict = compute_measured_building_max_dict(
                    measured_buildings_total_kw_pf_dict
                )

                # Free unused memory
                del measured_buildings_total_kw_pf_dict
                gc.collect()

                # Get lists of months, days, and hours from a sample dataframe in building_predicted_total_dict, used later to mask selected mdh
                months, days, hours = get_reference_calendar_arrays(building_predicted_total_dict)

                dss.Command(f'Redirect "{region_path}/Master.dss"')
                print(f"Redirected dss engine to {region_path}/Master.dss")

                # Organize all building_type entries under each feeder
                # for Loads.dss process
                feeder_dict = build_feeder_dict(building_predicted_total_dict)

                # Create a list of paths to original folders with
                # LineCodes.dss / Transformers.dss / Loads.dss
                folders_with_linecodes = file_ops.find_folders_with_file(
                    region_path,
                    "LineCodes.dss",
                    max_depth=3,
                )

                folders_with_transformers = file_ops.find_folders_with_file(
                    region_path,
                    "Transformers.dss",
                    max_depth=3,
                )

                # Get list of original feeder paths, folders with Loads.dss
                feeder_folders = file_ops.find_folders_with_file(
                    region_path,
                    "Loads.dss",
                )
                
                # Get list of original feeder paths, folders with PVSystems.dss
                folders_with_pvsystems = file_ops.find_folders_with_file(
                    region_path,
                    "PVSystems.dss",
                )


                # --- Iterate over all selected month-day-hour (mdh) ---
                for row_i in range(start_row_idx, end_row_idx):
                    mdh = city_inputs["list_of_mdh"][row_i]
                    m, d, h = mdh

                    print(f"Creating files for month:{m} hour:{d} day:{h}\n")

                    # Mask to select kw/kvar value for each building
                    # at the selected mdh
                    mdh_mask = (months == m) & (days == d) & (hours == h)

                    Ta = get_temperature_for_mdh(
                        city_inputs["TGW_weather_df"],
                        m,
                        d,
                        h,
                    )

                    print(
                        f"Found Temperature for TGW_scenario {TGW_scenario} "
                        f"TGW_weather_year {TGW_weather_year} {city} {region}: "
                        f"{Ta} (used for derating and resistance)\n"
                    )

                    create_thermal_files_for_mdh(
                        folders_with_linecodes=folders_with_linecodes,
                        folders_with_transformers=folders_with_transformers,
                        Ta=Ta,
                        Ta_near_worst=city_inputs["Ta_near_worst"],
                        T0=T0,
                        alpha_r=alpha_r,
                        f_amp_temp=f_amp_temp,
                        line_rating_mode=line_rating_mode,
                        transformers_rating_mode=transformers_rating_mode,
                        TGW_scenario=TGW_scenario,
                        TGW_weather_year=TGW_weather_year,
                        m=m,
                        d=d,
                        h=h,
                    )
                   
                    
                    create_loads_files_for_mdh(
                        feeder_dict=feeder_dict,
                        feeder_folders=feeder_folders,
                        building_predicted_total_dict=building_predicted_total_dict,
                        building_measured_max_dict=building_measured_max_dict,
                        TGW_scenario=TGW_scenario,
                        TGW_weather_year=TGW_weather_year,
                        m=m,
                        d=d,
                        h=h,
                        mdh_mask=mdh_mask,
                    )
                   
                    # Get solar irradiance at selected mdh
                    irradiance = get_irradiance_for_mdh(
                        city_inputs["TGW_weather_df"],
                        m,
                        d,
                        h,
                    )

                    print(
                        f"Found irradiance for TGW_scenario {TGW_scenario} "
                        f"TGW_weather_year {TGW_weather_year} {city} {region}: "
                        f"{irradiance} kW/m2 "
                        f"(used for PVSystems.dss irradiance)\n"
                    )

                    # Create PVSystems.dss with irradiance at selected mdh
                    new_pvsystems_paths = create_pvsystems_files_for_mdh(
                        folders_with_pvsystems=folders_with_pvsystems,
                        irradiance=irradiance,
                        TGW_scenario=TGW_scenario,
                        TGW_weather_year=TGW_weather_year,
                        m=m,
                        d=d,
                        h=h,
                    )
                    

                    # Create a duplicate master file with paths to new
                    # Loads.dss, LineCodes.dss, and Transformers.dss
                    new_master_file = create_master_file_for_mdh(
                        region_path=region_path,
                        solution_mode=solution_mode,
                        demand_mode=demand_mode,
                        line_rating_mode=line_rating_mode,
                        transformers_rating_mode=transformers_rating_mode,
                        TGW_scenario=TGW_scenario,
                        TGW_weather_year=TGW_weather_year,
                        m=m,
                        d=d,
                        h=h,
                    )

                    print(f"A modified master file was created in {new_master_file}\n")

                # Free unused memory after every region run
                del building_predicted_total_dict
                gc.collect()

                region_end_time = time.time()

                print(
                    f"---Runtime for {city} {region}: "
                    f"{(region_end_time - region_start_time) / 60:.2f} minutes---\n"
                )


end_time = time.time()
print(f"--- Total Runtime: {(end_time - start_time) / 60:.2f} minutes ---")